# 少数の例から同じカテゴリの語を増やす（ブートストラッピング）

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/22_lexical_semantics/bootstrapping.ipynb)

NLP クックブック 22-03 ／ 語の意味と意味の変化

「日本、フランス、中国」のような数個の例（シード）を与えるだけで、同じカテゴリの語をコーパスから集めたい。

In [1]:
!pip install -q datasets fugashi unidic-lite

In [2]:
import itertools, math, re
from collections import Counter
import numpy as np
import scipy.sparse as sp
from datasets import load_dataset
from fugashi import Tagger

ds = load_dataset("wikimedia/wikipedia", "20231101.ja", split="train", streaming=True)
articles = [ex["text"][:1500] for ex in itertools.islice(ds, 12000)]
tagger = Tagger()

# (名詞句, その前後の文脈パターン) の共起を数える。パターンは「左の語 X 右の語」と「X 右の語 その次の語」の 2 種類
cooc = Counter()
for a in articles:
    for s in re.split(r"(?<=。)|\n", a):
        if not 10 <= len(s) <= 150:
            continue
        c = []
        for w in tagger(s):
            if w.feature.pos1 == "名詞" and c and c[-1][1]:
                c[-1] = (c[-1][0] + w.surface, True)
            else:
                c.append((w.surface, w.feature.pos1 == "名詞"))
        c = [("<s>", False)] + c + [("</s>", False), ("</s>", False)]
        for i in range(1, len(c) - 2):
            if c[i][1] and len(c[i][0]) >= 2 and not c[i][0].isdigit():
                cooc[c[i][0], f"{c[i - 1][0]} X {c[i + 1][0]}"] += 1
                cooc[c[i][0], f"X {c[i + 1][0]} {c[i + 2][0]}"] += 1
print(len(cooc), "種類の共起")

2573274 種類の共起


In [3]:
# 行がパターン、列が語 (インスタンス) の行列を作る。要素は PMI (最大値で割って 0〜1 にする)
n_inst, n_pat = Counter(), Counter()
for (i, p), n in cooc.items():
    n_inst[i] += n; n_pat[p] += n
instances = [i for i, n in n_inst.items() if n >= 5]
patterns = [p for p, n in n_pat.items() if n >= 5]
inst_id, pat_id = {w: k for k, w in enumerate(instances)}, {w: k for k, w in enumerate(patterns)}
total, rows, cols, vals = sum(cooc.values()), [], [], []
for (i, p), n in cooc.items():
    if i in inst_id and p in pat_id:
        pmi = math.log2(n * total / (n_inst[i] * n_pat[p]))
        if pmi > 0:
            rows.append(pat_id[p]); cols.append(inst_id[i]); vals.append(pmi)
M = sp.csr_matrix((vals, (rows, cols)), shape=(len(patterns), len(instances)))
M = M / M.max()
print(M.shape, "非ゼロ要素", M.nnz)

(49837, 79844) 非ゼロ要素 1368381


In [4]:
def bootstrap(seeds, iterations=1):
    """シード → シードとよく共起するパターン → そのパターンとよく共起する語、を繰り返す"""
    inst_score = np.zeros(len(instances))
    inst_score[[inst_id[s] for s in seeds]] = 1.0
    for _ in range(iterations):
        pat_score = M @ inst_score; pat_score /= pat_score.max()        # パターンの信頼度: 信頼できる語と強く共起するほど高い
        inst_score = M.T @ pat_score; inst_score /= inst_score.max()    # 語の信頼度: 信頼できるパターンと強く共起するほど高い
    return inst_score, pat_score

for seeds in [["日本", "フランス", "中国"], ["ピアノ", "ギター", "ヴァイオリン"], ["夏目漱石", "森鴎外", "芥川龍之介"], ["サッカー", "野球", "テニス"]]:
    inst_score, pat_score = bootstrap(seeds)
    print("シード:", seeds)
    print("   パターン:", [patterns[k] for k in np.argsort(-pat_score)[:8]])
    print("   獲得した語:", [instances[k] for k in np.argsort(-inst_score) if instances[k] not in seeds][:18])

シード: ['日本', 'フランス', '中国']
   パターン: ['X が 世界最大', 'X に 返還', 'X 、 韓国', 'X の 天文', 'X の 古典', 'X 国 内', 'X の 南北朝時代', 'X の 詩人']
   獲得した語: ['イギリス', 'アメリカ', 'ドイツ', 'イタリア', 'ヨーロッパ', 'ロシア', 'スペイン', 'オランダ', 'インド', '世界', '多く', '韓国', 'ポルトガル', 'イングランド', 'ローマ', '米国', '外国', 'ギリシア']
シード: ['ピアノ', 'ギター', 'ヴァイオリン']
   パターン: ['X を 弾く', 'X を 弾き', 'X の 手ほどき', 'X 、 水泳', 'X の 腕前', 'X を 習い', 'X の 弦', 'X の 名手']
   獲得した語: ['音楽', 'オルガン', 'ボーカル', 'キーボード', 'ベース', 'ホルン', 'ミカン', 'ロック', 'アブラナ', 'ドラム', 'ナス', 'フルート', '数学', 'テレビ', '絵画', '演劇', '弦楽器', '野球']
シード: ['夏目漱石', '森鴎外', '芥川龍之介']
   パターン: ['X が 『', 'X の 小説', 'X の 『', '<s> X 『', 'X の 短編小説', 'X の 銅像', 'X の 自殺', 'X を 師']
   獲得した語: ['セルバンテス', '川端康成', '太宰治', 'アシモフ', '三島由紀夫', '本居宣長', '宮沢賢治', 'ヘーシオドス', 'ツヴァイク', 'ヘロドトス', '黒岩涙香', 'ビートルズ', '村上龍', 'トールキン', '法然', '司馬遷', '辻村深月', '手塚治虫']
シード: ['サッカー', '野球', 'テニス']
   パターン: ['X 界 の', 'X の クラブチーム', 'の X 好き', 'の X 界', 'X を 始め', 'X 場 ・', 'X 場 ）', 'X に 専念']
   獲得した語: ['競馬', 'スキー', '競技', '駐車', '飛行', 'スポーツ', '音楽', '映画', '電磁', '相撲', '国立競技',

- ブートストラッピングは「シードとよく一緒に出る文脈（パターン）を見つけ、そのパターンに当てはまる別の語を集める」ことを繰り返す。正解のラベルを数個しか必要としない。
- ここでの計算は Espresso というアルゴリズムを簡略化したもの。パターンの信頼度と語の信頼度を、PMI を重みにして交互に計算する。
- 「X を弾く」（楽器）、「X の小説」（作家）のように、カテゴリに特有のパターンが自動的に選ばれる。パターンを見ると、何を手がかりにしたかが分かる。
- 1 回の反復でも、かなり良い候補が得られる。ただし「音楽」「多く」のような、カテゴリの外の語も混ざる。
- 反復を増やせば候補は増えるが、次項のように別の問題（意味ドリフト）が起きる。